In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent / 'src'))

In [2]:
from typing import Any, List, Tuple

from data_handlers.echr_data_handler import EchrDataHandler
from utils.evaluation_utils import EvaluationUtils
from utils.project_utils import ProjectUtils

import textwrap

import pandas as pd

In [3]:
project_root: Path = ProjectUtils.get_project_root(); project_root

PosixPath('/home/ssaha/Projects/redacted-text-utility')

In [4]:
echr_data_handler: EchrDataHandler = EchrDataHandler(project_root)
echr_raw_file_names: List[str] = echr_data_handler.get_available_raw_files()

In [5]:
def evaluate_model(data_filename: str, 
                   model_names: List[str], 
                   data_fold_values: List[int], 
                   test_entity_percentile_ranges: List[Tuple[int, int]], 
                   replacement_strategies: List[str], 
                   export_path_root: Path) -> None:
    
    pe_df = echr_data_handler.get_private_entities_df(data_filename)
    id_column="itemid"
    text_column="text"
    class_column="binary_judgement"
    pe_column="text_pe_ontonotes5_ner-english-ontonotes-large"
    zero_entity_retain_text=True

    for model_name in model_names:

        for k in data_fold_values:
            
            data_dir_path = Path(f'/home/ssaha/model-checkpoints/echr/tc/{model_name}/additional-embeddings-none/sample-size-8435/data-fold-{k}')
            model_file_path = data_dir_path / 'learning-rate-5e-7' / 'max-epochs-25' / 'mini-batch-size-2' / 'best-model.pt'
            test_df = echr_data_handler.get_train_dev_test_datasetdict(k=k)["test"].to_pandas()
            pe_df_test = pe_df[pe_df['itemid'].isin(test_df['itemid'])]
            entity_counts_stat = pe_df_test['pe_count_total'].describe()
            entity_counts_stat = entity_counts_stat.astype(int)
            print()
            print(f"Statistics of private entities in test samples for data_fold={k}:\n{pd.DataFrame(entity_counts_stat).to_markdown()}")
            print()
            
            for test_entity_percentile_range in test_entity_percentile_ranges:
                lp, up = test_entity_percentile_range
                lower_bound = int(pe_df_test['pe_count_total'].quantile(lp/100))
                upper_bound = int(pe_df_test['pe_count_total'].quantile(up/100))
                filtered_itemids = pe_df_test[(pe_df_test['pe_count_total'] >= lower_bound) & (pe_df_test['pe_count_total'] <= upper_bound)]['itemid'].tolist()
                input_df = test_df[test_df['itemid'].isin(filtered_itemids)]
                number_of_samples = len(input_df)
                
                for replacement_strategy in replacement_strategies:
                    print()
                    print(textwrap.dedent(f"""
                        Evaluating for configuration:
                        -> model={model_name} 
                        -> data_fold={k}
                        -> test_entity_percentile_range={test_entity_percentile_range}
                        -> private_entity_count_range=[{lower_bound}, {upper_bound}]
                        -> total_items_in_range={number_of_samples}
                        -> replacement_strategy={replacement_strategy}
                    """).strip())
                    print()
                    result = EvaluationUtils.redact_and_evaluate_for_text_classifier(input_df=input_df,
                                                                                     pe_df=pe_df,
                                                                                     id_column=id_column,
                                                                                     text_column=text_column,
                                                                                     class_column=class_column,
                                                                                     pe_column=pe_column,
                                                                                     replacement_strategy=replacement_strategy,
                                                                                     zero_entity_retain_text=zero_entity_retain_text,
                                                                                     data_dir_path=data_dir_path,
                                                                                     model_file_path=model_file_path)
                    
                    export_path_dir = export_path_root / f'{model_name}' / f'{replacement_strategy}' / f'{test_entity_percentile_range[0]}-{test_entity_percentile_range[1]}'
                    export_path_dir.mkdir(parents=True, exist_ok=True)
                    export_path = export_path_dir / f'K{k}.txt'
                    with open(export_path, 'w') as f:
                        f.write(result.detailed_results)

In [6]:
data_filename = echr_raw_file_names[0]
model_names = ["xlm-roberta-large"] ## ["xlm-roberta-large", "bert-large-cased", "google--electra-large-discriminator"]
data_fold_values = [1] ## [1, 2, 3, 4, 5]
test_entity_percentile_ranges = [(0, 25)] ## [(0, 100), (0, 25), (25, 50), (50, 75), (75, 100)]
replacement_strategies = ["semantic_label_mask"] ## ["unredacted", "semantic_label_mask", "generic_mask", "random_mask"]
export_path_root = Path("/home/ssaha/Projects/echr_metrics")

evaluate_model(data_filename, model_names, data_fold_values, test_entity_percentile_ranges, replacement_strategies, export_path_root)

2026-01-21 17:10:13.415 | INFO     | data_handlers.echr_data_handler:get_dataframe_for_file:115 - Loading data from /home/ssaha/Projects/redacted-text-utility/data/raw/glnmario/ECHR/ECHR_Dataset.parquet



Statistics of private entities in test samples for data_fold=1:
|       |   pe_count_total |
|:------|-----------------:|
| count |             1686 |
| mean  |               85 |
| std   |               51 |
| min   |                7 |
| 25%   |               49 |
| 50%   |               72 |
| 75%   |              111 |
| max   |              357 |


Evaluating for configuration:
-> model=xlm-roberta-large 
-> data_fold=1
-> test_entity_percentile_range=(0, 25)
-> private_entity_count_range=[7, 49]
-> total_items_in_range=435
-> replacement_strategy=semantic_label_mask

2026-01-21 17:11:02,660 Reading data from /home/ssaha/model-checkpoints/echr/tc/xlm-roberta-large/additional-embeddings-none/sample-size-8435/data-fold-1
2026-01-21 17:11:02,663 Train: /home/ssaha/model-checkpoints/echr/tc/xlm-roberta-large/additional-embeddings-none/sample-size-8435/data-fold-1/train.csv
2026-01-21 17:11:02,663 Dev: /home/ssaha/model-checkpoints/echr/tc/xlm-roberta-large/additional-embeddings-none/

100%|██████████| 435/435 [01:00<00:00,  7.19it/s]


In [7]:
print(Path("/home/ssaha/Projects/echr_metrics/xlm-roberta-large/semantic_label_mask/0-25/K1.txt").read_text())


Results:
- F-score (micro) 0.8966
- F-score (macro) 0.8965
- Accuracy 0.8966

By class:
              precision    recall  f1-score   support

           1     0.8522    0.9469    0.8970       207
           0     0.9463    0.8509    0.8961       228

    accuracy                         0.8966       435
   macro avg     0.8993    0.8989    0.8965       435
weighted avg     0.9015    0.8966    0.8965       435

